In [ ]:
# Volcano Plot for IBD Paper

In [ ]:
import sys
import re
import pandas as pd
print(sys.version)

In [ ]:
### define setting to indicate raw data in saved files
database = "MGnify" # MGnify / IGC / Qin
grouping = "Groups" # Groups / Subgroups
new_discovery_study = "Werner" # ValdesMas / Werner
file_prefix = database +"_"+grouping + "_" + new_discovery_study + "_"
print(file_prefix)

In [ ]:
# Fig 1F (metaproteins identified in all discovery data sets after batch correction with MMUPHIN)
data = pd.read_csv(file_prefix+"shared_metaproteins(discovery_studies_normalized_MMUPHIN_corrected).csv", sep=",", index_col=0)#
#metaprotein_set = "shared_metaproteins_MMUPHIN_corrected"

# Fig 1G (metaproteins from 1F with variance explained by IBD > 0.2)
### read input from Variance analysis and only use the metaproteins provided by this file
variance_analysis_output = pd.read_csv("./NotebooksForUpload/MGnify_Groups_Werner_variance_explained_shared_MMUPHin.csv", sep=",", index_col=0)#
selected_proteins = list(variance_analysis_output["species"])
data = data.loc[selected_proteins,:]
metaprotein_set = "selected_metaproteins_MMUPHIN_corrected"


#data.rename(columns=lambda s: s.replace("P36_UCa", "P35_UCa"), inplace=True)

data.head(5)

In [ ]:
### load metadata
meta_data_path = r"SupplementaryFile1(WernerDiscovery)-Revision.xlsx"
meta_data = pd.read_excel(meta_data_path, sheet_name="SampleMetadata", index_col = "SampleID" )
print(meta_data.head(5))

### to make matching of sample names from meta data and mearurement data easier, replace all special characters by "_"
meta_data.columns = [re.sub(r'[^A-Za-z0-9]+', '_', col) for col in meta_data.columns]
data.columns = [re.sub(r'[^A-Za-z0-9]+', '_', col) for col in data.columns]


compare_panel = data
### join sample annotation and metaproteni abundance
for measurementID in compare_panel.columns:
    #print(type(measurementID))
    for sampleID in meta_data.columns:
        if sampleID in measurementID:
            compare_panel.loc["condition", measurementID]= meta_data.loc["condition", sampleID]
            compare_panel.loc["disease", measurementID]= meta_data.loc["disease", sampleID]
            compare_panel.loc["study", measurementID]= meta_data.loc["study", sampleID]
            compare_panel.loc["UseCase", measurementID]= meta_data.loc["UseCase", sampleID]
compare_panel

In [ ]:
# select only discovery columns
row = compare_panel.loc["UseCase"]

discovery_columns = row[row == 'BiomarkerDiscovery'].index
validation_columns = row[row == 'BiomarkerValidation'].index
specificity_columns = row[row == 'DiseaseSpecificity'].index

# Zeige nur die Spalten mit dem Wert "BiomarkerDiscovery", "BiomarkerValidation" oder "DiseaseSpecificity"
df_discovery = compare_panel[discovery_columns]
df_validation = compare_panel[validation_columns]
df_specificity = compare_panel[specificity_columns]

UseCaseSpecData={"discovery":df_discovery,"validation":df_validation,"specificity":df_specificity}

In [ ]:
# load # loading of data has to be evalu
annotation = pd.read_csv("IBDvsHealthy_annotated.csv", sep=";", index_col="#pg")#sharedProteins / selectedProteins
annotation.columns[0:35]
annotation[["members_identifier","task_1_Taxonomic_Annotation_Task_1_superkingdom","task_1_Taxonomic_Annotation_Task_1_species","GO_Term"]]

In [ ]:
LME_p_values = pd.read_csv("MGnify_Groups_Werner_MMUPHin_corrected_p-value(indidualMetaproteins).csv", index_col="Unnamed: 0")
LME_p_values

In [ ]:
import scipy.stats as stats
import numpy as np
# for each metaprotein use the columns with disease = IBD and disease = control
df_discovery.replace(["CD","UC","UCr","UCa","IBD-ND-OS","IBD-ND","UC-OS","CD-OS"],"IBD",inplace=True) # if all IBD case should be summarized
row = df_discovery.loc["disease"]
control_columns = row[row == 'normal'].index
#print(control_columns)
IBD_columns = row[row == 'IBD'].index
print(IBD_columns)
df_control = df_discovery[control_columns]
df_IBD = df_discovery[IBD_columns]

average_control=[]
average_diseased=[]
p_values =[]
annotations=[]
df_discovery= df_discovery.drop(["condition","disease","study","UseCase"])

for metaprotein in df_discovery.index:    
    control=np.asarray(df_control.loc[metaprotein]).astype(float)
    diseased=np.asarray(df_IBD.loc[metaprotein]).astype(float)

    # Benjamini-Hochberg corrected p-value from Linear mixed effect model analysis
    p_values.append(LME_p_values.loc[metaprotein,"Meta-Analysis"])
    average_diseased.append(np.average(diseased))
    average_control.append(np.average(control))
    
    #if "Homo sapiens" in str(annotation.loc[metaprotein,"task_1_Taxonomic_Annotation_Task_1_species"]):
    if "MGY" in str(annotation.loc[metaprotein,"members_identifier"]):
        annotations.append("Microbiome")
    else:
        annotations.append("Homo sapiens")

corr_p_values= stats.false_discovery_control(p_values, axis=0, method='bh')
df_discovery.insert(0,"average(diseased)", average_diseased)
df_discovery.insert(0,"average(control)", average_control)
df_discovery.insert(0,"p-value", corr_p_values)
df_discovery.insert(0,"TaxonomicAnnotation", annotations)

In [ ]:
# Calculate log2 fold change
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as lines

i=0
### protein group with ID=7964 only supported by one peptide --> drop
df_discovery = df_discovery.drop(7964, axis="rows")
dfList=[df_discovery]
for df in dfList:
    i+=1
    df['Log2FoldChange'] = np.log2(df.iloc[:, 3] / df.iloc[:, 2])

    # Calculate -log10(p-value)
    df['-log10_Pvalue'] = -np.log10(df.iloc[:, 1])

    # Set significance thresholds (you can adjust these as needed)
    fold_change_threshold = 1.0
    significance_threshold = -np.log10(0.05)
    #xlim1=max(abs())

    # Filter significant features
    significant_df = df[(abs(df['Log2FoldChange']) > fold_change_threshold) & (df['-log10_Pvalue'] > significance_threshold)]

    # Select the human proteins
    human_df = df[abs(df['TaxonomicAnnotation'] == "Homo sapiens")]

        #plt.xlim=
    #print(max(abs(significant_df['Log2FoldChange'])))
    #print(abs(min(np.array(significant_df['Log2FoldChange'])))
    a = np.array(df['Log2FoldChange'])
    min_value = a[np.isfinite(a)].min()
    max_value = a[np.isfinite(a)].max()
    xlimit=max([abs(min_value),abs(max_value)])*1.2
    print(xlimit)
    print(min_value)
    print(max_value)
    
    # Plotting the volcano plot
    plt.figure(figsize=(10, 6))
    plt.xlim(-xlimit,xlimit)
    plt.ylim(-0,6)
    plt.scatter(df['Log2FoldChange'], df['-log10_Pvalue'], color='green')
    plt.scatter(human_df['Log2FoldChange'], human_df['-log10_Pvalue'], color='red', label='p-value <0.05 and log2 Fold-Change >2')
    
    
    plt.axhline(y=-np.log10(0.05), color='black', linestyle='--')
    plt.axvline(x=fold_change_threshold, color='black', linestyle='--')
    plt.axvline(x=-fold_change_threshold, color='black', linestyle='--')

    line1 = lines.Line2D([], [], color="white", marker='o', markerfacecolor="red", markersize = 10)
    line2 = lines.Line2D([], [], color="white", marker='o', markerfacecolor="green", markersize = 10)
    plt.legend((line1, line2), ('Human Metaproteins', 'Microbial Metaproteins'), fontsize = 15, numpoints=1, loc=0)

    
    plt.xticks(fontsize= 15)
    plt.yticks(fontsize= 15)
    #plt.title('Volcano Plot')
    plt.xlabel('Log2 Fold Change', fontsize=15)
    plt.ylabel('-log10(P-value)',fontsize=15)
    #plt.legend()
    
    plt.savefig(f'VolcanoPlot{i}.png', dpi=500)
    plt.show()